# Gold Company Hiring Mart

This notebook creates the company-level hiring activity data mart with daily granularity.

## Architecture

**Input**: `workspace.warehouse.fact_job_postings`  
**Output**: `workspace.gold.gold_company_hiring`  
**Mode**: Full refresh (CREATE OR REPLACE)

## Schema

**Grain**: Company × Date  
**Primary Key**: (company_sk, hiring_date_sk)

**Key Metrics**:
- `active_jobs`: Active job count
- `new_jobs_30d`: New jobs last 30 days
- `top_role`: Most hired role
- `top_location`: Primary hiring location
- `remote_ratio`: Ratio of remote jobs

In [0]:
# Databricks notebook parameters
dbutils.widgets.text("lookback_days", "365", "Lookback Days")
dbutils.widgets.dropdown("force_full_refresh", "false", ["true", "false"], "Force Full Refresh")

# Get parameter values
lookback_days = int(dbutils.widgets.get("lookback_days"))
force_full_refresh = dbutils.widgets.get("force_full_refresh").lower() == "true"

In [0]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

# Configuration
CATALOG = "workspace"
WAREHOUSE_SCHEMA = f"{CATALOG}.warehouse"
GOLD_SCHEMA = f"{CATALOG}.gold"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Current run metadata
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = datetime.now()
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Run ID: {run_id}")
print(f"Lookback days: {lookback_days}")
print(f"Cutoff date: {cutoff_date}")
print(f"Force full refresh: {force_full_refresh}")

In [0]:
# Create metadata table to track refresh runs
metadata_table = f"{METADATA_SCHEMA}.gold_company_hiring_refresh_log"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {metadata_table} (
  run_id STRING,
  run_timestamp TIMESTAMP,
  lookback_days INT,
  cutoff_date INT,
  rows_created BIGINT,
  unique_companies BIGINT,
  unique_dates BIGINT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DOUBLE,
  status STRING,
  error_message STRING
)
USING DELTA
COMMENT 'Tracks refresh runs for gold_company_hiring'
""")

print(f"Metadata table ready: {metadata_table}")

In [0]:
import time

start_time = time.time()

print("Creating gold_company_hiring mart...")

# Execute CREATE OR REPLACE
spark.sql(f"""
CREATE OR REPLACE TABLE {GOLD_SCHEMA}.gold_company_hiring
USING DELTA
COMMENT 'Company-level hiring activity and metrics'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
)
AS

WITH base_daily AS (
  SELECT
    f.company_sk,
    f.posting_date_sk AS hiring_date_sk,
    f.role_sk,
    f.location_sk,
    j.remote_type,
    COUNT(CASE WHEN f.active_flag = TRUE THEN 1 END) AS active_jobs,
    COUNT(*) AS total_jobs
  FROM {WAREHOUSE_SCHEMA}.fact_job_postings f
  LEFT JOIN {WAREHOUSE_SCHEMA}.dim_job j ON f.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE f.posting_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), {lookback_days}), 'yyyyMMdd') AS INT)
  GROUP BY f.company_sk, f.posting_date_sk, f.role_sk, f.location_sk, j.remote_type
),

jobs_30d AS (
  SELECT
    company_sk,
    hiring_date_sk,
    SUM(total_jobs) AS new_jobs_30d
  FROM base_daily
  WHERE hiring_date_sk >= CAST(DATE_FORMAT(DATE_SUB(CURRENT_DATE(), 30), 'yyyyMMdd') AS INT)
  GROUP BY company_sk, hiring_date_sk
),

top_roles AS (
  SELECT
    company_sk,
    hiring_date_sk,
    role_sk,
    SUM(active_jobs) AS role_jobs,
    ROW_NUMBER() OVER (PARTITION BY company_sk, hiring_date_sk ORDER BY SUM(active_jobs) DESC, role_sk) AS rn
  FROM base_daily
  GROUP BY company_sk, hiring_date_sk, role_sk
),

top_locations AS (
  SELECT
    company_sk,
    hiring_date_sk,
    location_sk,
    SUM(active_jobs) AS location_jobs,
    ROW_NUMBER() OVER (PARTITION BY company_sk, hiring_date_sk ORDER BY SUM(active_jobs) DESC, location_sk) AS rn
  FROM base_daily
  GROUP BY company_sk, hiring_date_sk, location_sk
),

remote_stats AS (
  SELECT
    company_sk,
    hiring_date_sk,
    SUM(CASE WHEN remote_type = 'REMOTE' THEN active_jobs ELSE 0 END) AS remote_jobs,
    SUM(active_jobs) AS total_active
  FROM base_daily
  GROUP BY company_sk, hiring_date_sk
),

aggregated AS (
  SELECT
    bd.company_sk,
    bd.hiring_date_sk,
    SUM(bd.active_jobs) AS active_jobs,
    COALESCE(j30.new_jobs_30d, 0) AS new_jobs_30d,
    FIRST(dr.role_name) AS top_role,
    FIRST(dl.location_name) AS top_location,
    CASE 
      WHEN rs.total_active > 0 THEN CAST(CAST(rs.remote_jobs AS DOUBLE) / CAST(rs.total_active AS DOUBLE) AS DECIMAL(5,4))
      ELSE NULL
    END AS remote_ratio
  FROM base_daily bd
  LEFT JOIN jobs_30d j30 ON bd.company_sk = j30.company_sk AND bd.hiring_date_sk = j30.hiring_date_sk
  LEFT JOIN top_roles tr ON bd.company_sk = tr.company_sk AND bd.hiring_date_sk = tr.hiring_date_sk AND tr.rn = 1
  LEFT JOIN {WAREHOUSE_SCHEMA}.dim_role dr ON tr.role_sk = dr.role_sk
  LEFT JOIN top_locations tl ON bd.company_sk = tl.company_sk AND bd.hiring_date_sk = tl.hiring_date_sk AND tl.rn = 1
  LEFT JOIN {WAREHOUSE_SCHEMA}.dim_location dl ON tl.location_sk = dl.location_sk
  LEFT JOIN remote_stats rs ON bd.company_sk = rs.company_sk AND bd.hiring_date_sk = rs.hiring_date_sk
  GROUP BY bd.company_sk, bd.hiring_date_sk, j30.new_jobs_30d, rs.total_active, rs.remote_jobs
)

SELECT
  company_sk,
  hiring_date_sk,
  active_jobs,
  new_jobs_30d,
  top_role,
  top_location,
  remote_ratio,
  CURRENT_TIMESTAMP() AS updated_at
FROM aggregated
ORDER BY hiring_date_sk DESC, active_jobs DESC
""")

processing_time = time.time() - start_time
print(f"✓ Table created successfully in {processing_time:.2f} seconds")

In [0]:
# Get table statistics
table_stats = spark.sql(f"""
SELECT
  COUNT(*) AS rows_created,
  COUNT(DISTINCT company_sk) AS unique_companies,
  COUNT(DISTINCT hiring_date_sk) AS unique_dates
FROM {GOLD_SCHEMA}.gold_company_hiring
""").collect()[0]

rows_created = table_stats["rows_created"]
unique_companies = table_stats["unique_companies"]
unique_dates = table_stats["unique_dates"]

# Log metadata - match existing table schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, DoubleType, TimestampType, BooleanType

metadata_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("lookback_days", IntegerType(), True),
    StructField("cutoff_date", IntegerType(), True),
    StructField("rows_created", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("unique_companies", LongType(), True),
    StructField("unique_sectors", LongType(), True),
    StructField("force_full_refresh", BooleanType(), True),
    StructField("processing_time_seconds", DoubleType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("status", StringType(), True)
])

metadata_df = spark.createDataFrame([(
    run_id,
    lookback_days,
    cutoff_date,
    rows_created,
    0,  # rows_updated (not applicable for full refresh)
    unique_companies,
    unique_dates,  # using unique_dates for unique_sectors column
    force_full_refresh,
    processing_time,
    run_timestamp,
    "success"
)], schema=metadata_schema)

metadata_df.write.format("delta").mode("append").saveAsTable(metadata_table)

print(f"\nRefresh Summary:")
print(f"  Run ID: {run_id}")
print(f"  Rows created: {rows_created:,}")
print(f"  Unique companies: {unique_companies}")
print(f"  Unique dates: {unique_dates}")
print(f"  Processing time: {processing_time:.2f}s")

In [0]:
%sql
-- Validation: Summary Statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT company_sk) AS unique_companies,
  COUNT(DISTINCT hiring_date_sk) AS unique_dates,
  MIN(hiring_date_sk) AS earliest_date,
  MAX(hiring_date_sk) AS latest_date,
  SUM(active_jobs) AS total_active_jobs,
  SUM(new_jobs_30d) AS total_jobs_30d,
  AVG(remote_ratio) AS avg_remote_ratio,
  MAX(updated_at) AS last_refreshed
FROM workspace.gold.gold_company_hiring;